# Analisis SUS (System Usability Scale) -- aplicado, con un hallazgo de integridad en las fechas

**Estado real (ver `docs/mediciones/sus/SUS-RESULTS.md`, 2026-09-17): el instrumento SUS se aplico a 15**
**personas, pero solo 4 tienen fecha de aplicacion verificable (2026-09-17 o anterior).** Las otras 11
hojas tienen una fecha escrita a mano posterior al commit que las versiona -- y en varios casos, posterior
al dia de hoy, lo cual es imposible para una hoja fisica ya firmada. Se tratan como pendientes de
confirmar, no como datos validos ni como datos descartados. Una version anterior del proyecto afirmaba
ademas un puntaje fabricado con 10 evaluadores que nunca existieron -- ese dato se retiro. Este notebook
implementa la formula de calculo de Brooke (1996) como funciones reutilizables, las verifica con un
autotest sintetico (claramente marcado como tal, no como resultados de participantes), y calcula el
resultado de cierre solo sobre las 4 respuestas con fecha verificable.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'scripts' else os.getcwd()
RESPONSES_CSV = os.path.join(REPO_ROOT, 'docs', 'mediciones', 'sus', 'sus-respuestas.csv')

## Formula de puntaje SUS (Brooke, 1996)

10 preguntas, escala Likert 1-5. Impares (1,3,5,7,9): `valor - 1`. Pares (2,4,6,8,10): `5 - valor`.
Suma de las 10 x 2.5 = puntaje SUS (0-100) por participante. El puntaje del estudio es el promedio.

In [2]:
def sus_score(respuestas):
    '''respuestas: lista/array de 10 enteros (1-5), preguntas Q1..Q10 en orden.'''
    respuestas = np.asarray(respuestas, dtype=float)
    if respuestas.shape[-1] != 10:
        raise ValueError('Se esperan exactamente 10 respuestas por participante')
    odd_idx = [0, 2, 4, 6, 8]   # Q1,Q3,Q5,Q7,Q9 (0-indexado)
    even_idx = [1, 3, 5, 7, 9]  # Q2,Q4,Q6,Q8,Q10
    puntos = np.zeros_like(respuestas)
    puntos[..., odd_idx] = respuestas[..., odd_idx] - 1
    puntos[..., even_idx] = 5 - respuestas[..., even_idx]
    return puntos.sum(axis=-1) * 2.5


def sus_grade(score):
    '''Escala de adjetivos de Bangor et al. (2009), de uso comun junto al puntaje SUS.'''
    if score >= 84.1: return 'A+ (Excelente)'
    if score >= 80.8: return 'A'
    if score >= 78.9: return 'A-'
    if score >= 77.2: return 'B+'
    if score >= 74.1: return 'B'
    if score >= 72.6: return 'B-'
    if score >= 71.1: return 'C+'
    if score >= 65.0: return 'C'
    if score >= 62.7: return 'C-'
    if score >= 51.7: return 'D'
    return 'F'


## Autotest de la funcion (datos sinteticos de verificacion, NO datos de participantes)

Verifica que `sus_score` reproduce el ejemplo canonico de la literatura: respuestas todas neutras (3)
deben dar 50/100 (el punto medio de la escala), y respuestas maximas favorables deben dar 100/100.

In [3]:
# Caso 1: todas las respuestas neutras (3) -> debe dar exactamente 50.0
neutro = [3] * 10
assert sus_score(neutro) == 50.0, f'esperado 50.0, obtuvo {sus_score(neutro)}'

# Caso 2: respuestas maximamente favorables (impares=5, pares=1) -> debe dar 100.0
favorable = [5, 1, 5, 1, 5, 1, 5, 1, 5, 1]
assert sus_score(favorable) == 100.0, f'esperado 100.0, obtuvo {sus_score(favorable)}'

# Caso 3: respuestas maximamente desfavorables (impares=1, pares=5) -> debe dar 0.0
desfavorable = [1, 5, 1, 5, 1, 5, 1, 5, 1, 5]
assert sus_score(desfavorable) == 0.0, f'esperado 0.0, obtuvo {sus_score(desfavorable)}'

print('Autotest OK: la funcion sus_score reproduce los 3 casos canonicos (0, 50, 100).')


Autotest OK: la funcion sus_score reproduce los 3 casos canonicos (0, 50, 100).


## Carga de respuestas reales

Formato real de `docs/mediciones/sus/sus-respuestas.csv`: una fila por participante, columnas
`participante,rol,fecha,fecha_verificable,q1,q2,...,q10,sus_score` (valores 1-5 en q1..q10;
`fecha_verificable` es `si`/`no`). Esta celda filtra por `fecha_verificable == 'si'` antes de calcular el
resultado del grupo, para no mezclar las respuestas con fecha confirmada con las que todavia no la
tienen -- ver la celda de estado arriba y `docs/mediciones/sus/SUS-RESULTS.md` para el detalle completo
del hallazgo.

In [ ]:
if not os.path.exists(RESPONSES_CSV):
    print(f'[pendiente] {RESPONSES_CSV} no existe -- ver docs/mediciones/sus/SUS-RESULTS.md.')
else:
    df = pd.read_csv(RESPONSES_CSV)
    preguntas = [f'q{i}' for i in range(1, 11)]
    df['sus_score_calc'] = sus_score(df[preguntas].values)
    assert np.allclose(df['sus_score_calc'], df['sus_score']), 'el puntaje recalculado no coincide con el CSV'
    df['grado'] = df['sus_score_calc'].apply(sus_grade)
    print(f"Total de hojas recolectadas: {len(df)}")
    print(df[['participante', 'rol', 'fecha', 'fecha_verificable', 'sus_score_calc', 'grado']].to_string(index=False))

    validos = df[df['fecha_verificable'] == 'si']
    pendientes = df[df['fecha_verificable'] != 'si']
    print(f"\nCon fecha verificable: {len(validos)} -- pendientes de confirmar: {len(pendientes)}")

    n = len(validos)
    scores = validos['sus_score_calc'].values
    media = scores.mean()
    de = scores.std(ddof=1)
    se = de / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, df=n - 1)
    margen = tcrit * se
    print(f"\nResultado de cierre (solo fecha verificable, n={n}): "
          f"media={media:.2f} DE={de:.2f} IC95=[{media - margen:.2f}, {media + margen:.2f}]")
